# 7. Upload MP4 videos to R2 and load annotation exports into Supabase bronze

This notebook converts each full source AVI to an H.264 MP4, uploads it to the public Cloudflare R2 `annotationbronze` bucket, deletes the old `public` tables, and loads notebook 06 exports into the Supabase `bronze` schema. Each video URL is saved in `bronze.videos`. Videos already present in R2 at the stable `original-videos/<video_id>.mp4` key are reused and not re-uploaded.

Before running, copy `.env.example` to `.env` and set:

- `SUPABASE_DB_URL` — PostgreSQL connection URL with table-creation permissions.
- `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` — write credentials for the R2 bucket; never commit them.
- `R2_ENDPOINT_URL`, `R2_BUCKET`, and `R2_PUBLIC_URL` — the R2 S3 endpoint, bucket, and public domain.

Every run overwrites the five Bronze tables using stable IDs. R2 video objects are reused when `original-videos/<video_id>.mp4` already exists, so those files are not converted or uploaded again.

- `bronze.videos` 1→N `bronze.events`
- `bronze.videos` 1→N `bronze.objects`
- `bronze.events` 1→N `bronze.event_objects` N←1 `bronze.objects` (an event can involve many objects; a tracked object can appear in more than one event)
- `bronze.objects` 1→N `bronze.geometries`

In [1]:
import hashlib
import json
import os
import subprocess
from threading import Lock
from time import perf_counter
from pathlib import Path
from urllib.parse import quote

import boto3
from boto3.s3.transfer import TransferConfig
from botocore.config import Config
from botocore.exceptions import ClientError
import psycopg
from psycopg.types.json import Jsonb
from dotenv import load_dotenv

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "notebooks").is_dir() and (parent / "annotation_pipeline").is_dir()
)
RAW_VIDEOS = PROJECT_ROOT / "data" / "raw" / "videos"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MP4_DIR = PROCESSED_DIR / "source_mp4"
BATCH_SIZE = 500

load_dotenv(PROJECT_ROOT / ".env")
SUPABASE_DB_URL = os.getenv("SUPABASE_DB_URL")
R2_ACCESS_KEY_ID = os.getenv("R2_ACCESS_KEY_ID")
R2_SECRET_ACCESS_KEY = os.getenv("R2_SECRET_ACCESS_KEY")
R2_ENDPOINT_URL = os.getenv("R2_ENDPOINT_URL")
R2_BUCKET = os.getenv("R2_BUCKET", "annotationbronze")
R2_PUBLIC_URL = os.getenv("R2_PUBLIC_URL")
if R2_ENDPOINT_URL and R2_ENDPOINT_URL.rstrip("/").endswith(f"/{R2_BUCKET}"):
    R2_ENDPOINT_URL = R2_ENDPOINT_URL.rstrip("/").removesuffix(f"/{R2_BUCKET}")

missing = [
    name
    for name, value in {
        "SUPABASE_DB_URL": SUPABASE_DB_URL,
        "R2_ACCESS_KEY_ID": R2_ACCESS_KEY_ID,
        "R2_SECRET_ACCESS_KEY": R2_SECRET_ACCESS_KEY,
        "R2_ENDPOINT_URL": R2_ENDPOINT_URL,
        "R2_PUBLIC_URL": R2_PUBLIC_URL,
    }.items()
    if not value
]
if missing:
    raise EnvironmentError(f"Set {', '.join(missing)} in {PROJECT_ROOT / '.env'} before running.")


SCHEMA_SQL = """
CREATE SCHEMA IF NOT EXISTS bronze;

DROP TABLE IF EXISTS public.geometries, public.events, public.objects, public.videos CASCADE;
DROP TABLE IF EXISTS bronze.geometries, bronze.event_objects, bronze.events, bronze.objects, bronze.videos CASCADE;

CREATE TABLE IF NOT EXISTS bronze.videos (
    video_id TEXT PRIMARY KEY,
    video_name TEXT NOT NULL UNIQUE,
    camera_id TEXT,
    scene TEXT,
    capture_start_local TIMESTAMP,
    capture_end_local TIMESTAMP,
    capture_time_zone TEXT,
    duration_seconds DOUBLE PRECISION,
    frame_count INTEGER,
    fps DOUBLE PRECISION,
    frame_width INTEGER,
    frame_height INTEGER,
    source_video_path TEXT,
    storage_bucket TEXT,
    storage_path TEXT UNIQUE,
    video_url TEXT,
    source_metadata JSONB NOT NULL,
    created_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    updated_at TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS bronze.objects (
    object_id TEXT PRIMARY KEY,
    source_object_id TEXT NOT NULL,
    detection_count INTEGER NOT NULL,
    first_seen_seconds DOUBLE PRECISION,
    last_seen_seconds DOUBLE PRECISION,
    label_details JSONB NOT NULL
);

CREATE TABLE IF NOT EXISTS bronze.events (
    event_id TEXT PRIMARY KEY,
    video_id TEXT NOT NULL REFERENCES bronze.videos(video_id) ON DELETE CASCADE,
    event_name TEXT,
    description TEXT,
    start_frame INTEGER,
    end_frame INTEGER,
    start_seconds DOUBLE PRECISION,
    end_seconds DOUBLE PRECISION,
    source_status TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS bronze.event_objects (
    event_id TEXT NOT NULL REFERENCES bronze.events(event_id) ON DELETE CASCADE,
    object_id TEXT NOT NULL REFERENCES bronze.objects(object_id) ON DELETE CASCADE,
    PRIMARY KEY (event_id, object_id)
);

CREATE TABLE IF NOT EXISTS bronze.geometries (
    geometry_id TEXT PRIMARY KEY,
    object_id TEXT NOT NULL REFERENCES bronze.objects(object_id) ON DELETE CASCADE,
    source_geometry_id INTEGER NOT NULL,
    frame_index INTEGER NOT NULL,
    timestamp_seconds DOUBLE PRECISION NOT NULL,
    label TEXT,
    confidence DOUBLE PRECISION,
    spatial_position TEXT,
    bounding_box_pixels JSONB NOT NULL
);

ALTER TABLE bronze.geometries DROP CONSTRAINT IF EXISTS geometries_object_source_geometry_key;
ALTER TABLE bronze.geometries ADD CONSTRAINT geometries_object_source_geometry_key
    UNIQUE (object_id, source_geometry_id);

CREATE INDEX IF NOT EXISTS events_video_id_idx ON bronze.events(video_id);
CREATE INDEX IF NOT EXISTS event_objects_object_id_idx ON bronze.event_objects(object_id);
CREATE INDEX IF NOT EXISTS geometries_object_id_idx ON bronze.geometries(object_id);
"""


def stable_id(*parts):
    """Return a globally stable SHA-256 identifier from namespaced values."""
    return hashlib.sha256(":".join(map(str, parts)).encode("utf-8")).hexdigest()


def batches(items, size=BATCH_SIZE):
    for start in range(0, len(items), size):
        yield items[start : start + size]


def find_source_video(video_name):
    candidates = [RAW_VIDEOS / f"{video_name}.r13.avi", RAW_VIDEOS / f"{video_name}.avi"]
    return next((path for path in candidates if path.is_file()), None)


def r2_client():
    """Return a boto3 S3 client configured for Cloudflare R2."""
    return boto3.client(
        "s3",
        endpoint_url=R2_ENDPOINT_URL,
        aws_access_key_id=R2_ACCESS_KEY_ID,
        aws_secret_access_key=R2_SECRET_ACCESS_KEY,
        region_name="auto",
        config=Config(
            signature_version="s3v4",
            connect_timeout=30,
            read_timeout=900,
            retries={"max_attempts": 5, "mode": "standard"},
        ),
    )


def r2_public_url(storage_path):
    return f"{R2_PUBLIC_URL.rstrip('/')}/{quote(storage_path, safe='/')}"


def r2_object_exists(client, storage_path):
    """Return True when a non-empty object already exists at this R2 key."""
    try:
        response = client.head_object(Bucket=R2_BUCKET, Key=storage_path)
    except ClientError as error:
        status = error.response.get("ResponseMetadata", {}).get("HTTPStatusCode")
        code = str(error.response.get("Error", {}).get("Code", ""))
        if status in {404, 403} or code in {"404", "403", "NoSuchKey", "NotFound"}:
            listed = client.list_objects_v2(Bucket=R2_BUCKET, Prefix=storage_path, MaxKeys=10)
            return any(
                obj.get("Key") == storage_path and int(obj.get("Size") or 0) > 0
                for obj in listed.get("Contents") or []
            )
        raise
    return int(response.get("ContentLength") or 0) > 0


def convert_to_mp4(source_video, output_path):
    """Create a browser-compatible H.264 MP4 from the original full AVI."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.is_file() and output_path.stat().st_mtime >= source_video.stat().st_mtime:
        print(f"  Reusing MP4: {output_path.name}", flush=True)
        return output_path
    print(f"  Converting AVI to MP4: {source_video.name}", flush=True)
    subprocess.run(
        [
            "ffmpeg", "-y", "-i", str(source_video), "-map", "0:v:0", "-map", "0:a?",
            "-c:v", "libx264", "-crf", "20", "-preset", "medium", "-c:a", "aac",
            "-movflags", "+faststart", str(output_path),
        ],
        check=True,
    )
    print(f"  MP4 ready: {output_path.name}", flush=True)
    return output_path


class UploadProgress:
    """Print multipart upload progress in 10-percent increments."""

    def __init__(self, filename, total_bytes):
        self.filename = filename
        self.total_bytes = total_bytes
        self.uploaded_bytes = 0
        self.next_percent = 10
        self.lock = Lock()

    def __call__(self, bytes_transferred):
        with self.lock:
            self.uploaded_bytes += bytes_transferred
            percent = min(100, int(self.uploaded_bytes * 100 / self.total_bytes))
            while percent >= self.next_percent:
                print(f"  R2 upload {self.filename}: {self.next_percent}%", flush=True)
                self.next_percent += 10


def upload_video_to_r2(storage_path, mp4_path, client=None):
    """Upload an MP4 to R2, or reuse the existing object when the key is already present."""
    client = client or r2_client()
    if r2_object_exists(client, storage_path):
        print(f"  Reusing existing R2 object: {storage_path}", flush=True)
        return r2_public_url(storage_path)
    file_size_mib = mp4_path.stat().st_size / 1024**2
    print(f"  Uploading {mp4_path.name} to R2 ({file_size_mib:.1f} MiB)", flush=True)
    client.upload_file(
        str(mp4_path),
        R2_BUCKET,
        storage_path,
        ExtraArgs={"ContentType": "video/mp4"},
        Callback=UploadProgress(mp4_path.name, mp4_path.stat().st_size),
        Config=TransferConfig(
            multipart_threshold=8 * 1024**2,
            multipart_chunksize=8 * 1024**2,
            max_concurrency=4,
        ),
    )
    print(f"  R2 upload complete: {storage_path}", flush=True)
    return r2_public_url(storage_path)


def load_exports(metadata_path):
    video_name = metadata_path.name.removesuffix(".metadata.json")
    export_dir = metadata_path.parent
    paths = {
        "metadata": metadata_path,
        "events": export_dir / f"{video_name}.activities.json",
        "objects": export_dir / f"{video_name}.types.json",
        "geometries": export_dir / f"{video_name}.geom.json",
    }
    missing_paths = [str(path) for path in paths.values() if not path.is_file()]
    if missing_paths:
        raise FileNotFoundError("Missing export files:\n" + "\n".join(missing_paths))
    return video_name, {name: json.loads(path.read_text(encoding="utf-8")) for name, path in paths.items()}


def create_tables():
    print("Creating the bronze schema and tables; removing legacy public tables...", flush=True)
    with psycopg.connect(SUPABASE_DB_URL) as connection:
        with connection.cursor() as cursor:
            cursor.execute(SCHEMA_SQL)
        connection.commit()


def reset_tables():
    """Replace all pipeline data without dropping the table definitions or indexes."""
    print("Clearing existing bronze table rows...", flush=True)
    with psycopg.connect(SUPABASE_DB_URL) as connection:
        with connection.cursor() as cursor:
            cursor.execute(
                "TRUNCATE TABLE bronze.geometries, bronze.event_objects, bronze.events, bronze.objects, bronze.videos CASCADE"
            )
        connection.commit()


JSON_COLUMNS = {
    "videos": {"source_metadata"},
    "objects": {"label_details"},
    "geometries": {"bounding_box_pixels"},
}


def insert_rows(table_name, rows):
    """Insert records into the bronze schema, preserving nested JSON fields."""
    if not rows:
        return
    print(f"  Inserting {len(rows):,} rows into bronze.{table_name}...", flush=True)
    columns = list(rows[0])
    statement = (
        f"INSERT INTO bronze.{table_name} ({', '.join(columns)}) "
        f"VALUES ({', '.join('%s' for _ in columns)})"
    )
    values = [
        tuple(
            Jsonb(row[column]) if column in JSON_COLUMNS.get(table_name, set()) else row[column]
            for column in columns
        )
        for row in rows
    ]
    with psycopg.connect(SUPABASE_DB_URL) as connection:
        with connection.cursor() as cursor:
            written = 0
            for batch_number, batch in enumerate(batches(values), start=1):
                cursor.executemany(statement, batch)
                written += len(batch)
                if batch_number % 5 == 0 or written == len(values):
                    print(f"    {table_name}: {written:,}/{len(values):,} rows", flush=True)
        connection.commit()


def validate_exports(video_name, exports):
    """Validate all local references before replacing data in Supabase."""
    if exports["metadata"].get("video_id") != stable_id("video", video_name):
        raise ValueError(f"Unexpected video_id in metadata export for {video_name}")
    object_ids = {item["object_id"] for item in exports["objects"]}
    event_actor_ids = {
        actor["object_id"] for event in exports["events"] for actor in event["actors"]
    }
    geometry_object_ids = {item["object_id"] for item in exports["geometries"]}
    unknown_ids = (event_actor_ids | geometry_object_ids) - object_ids
    if unknown_ids:
        raise ValueError(f"{video_name} references unknown objects: {sorted(unknown_ids)}")


def sync_video(video_name, exports):
    metadata = exports["metadata"]
    video_id = metadata.get("video_id")
    if video_id != stable_id("video", video_name):
        raise ValueError(f"Unexpected video_id in metadata export for {video_name}")

    storage_path = f"original-videos/{video_id}.mp4"
    client = r2_client()
    if r2_object_exists(client, storage_path):
        print(f"  R2 already has {storage_path}; skipping convert and upload", flush=True)
        video_url = r2_public_url(storage_path)
    else:
        source_video = find_source_video(video_name)
        if source_video is None:
            raise FileNotFoundError(f"Original AVI not found for {video_name} in {RAW_VIDEOS}")
        mp4_path = convert_to_mp4(source_video, MP4_DIR / f"{video_name}.mp4")
        video_url = upload_video_to_r2(storage_path, mp4_path, client=client)

    video_row = {
        "video_id": video_id,
        "video_name": video_name,
        "camera_id": metadata.get("camera_id"),
        "scene": metadata.get("scene"),
        "capture_start_local": metadata.get("capture_start_local"),
        "capture_end_local": metadata.get("capture_end_local"),
        "capture_time_zone": metadata.get("capture_time_zone"),
        "duration_seconds": metadata.get("duration_seconds"),
        "frame_count": metadata.get("frame_count"),
        "fps": metadata.get("fps"),
        "frame_width": metadata.get("frame_width"),
        "frame_height": metadata.get("frame_height"),
        "source_video_path": metadata.get("source_video_path"),
        "storage_bucket": R2_BUCKET,
        "storage_path": storage_path,
        "video_url": video_url,
        "source_metadata": metadata,
    }
    insert_rows("videos", [video_row])

    object_id_map = {
        item["object_id"]: stable_id("object", video_id, item["object_id"])
        for item in exports["objects"]
    }
    object_rows = [
        {
            "object_id": object_id_map[item["object_id"]],
            "source_object_id": item["object_id"],
            "detection_count": item["detection_count"],
            "first_seen_seconds": item.get("first_seen_seconds"),
            "last_seen_seconds": item.get("last_seen_seconds"),
            "label_details": item["labels"],
        }
        for item in exports["objects"]
    ]
    insert_rows("objects", object_rows)

    event_rows = []
    event_object_rows = []
    for item in exports["events"]:
        actor_ids = [actor["object_id"] for actor in item["actors"]]
        unknown_ids = set(actor_ids) - object_id_map.keys()
        if unknown_ids:
            raise ValueError(f"Event {item['event_id']} references unknown objects: {sorted(unknown_ids)}")
        event_rows.append(
            {
                "event_id": item["event_id"],
                "video_id": video_id,
                "event_name": item.get("event_name"),
                "description": item.get("description"),
                "start_frame": item.get("start_frame"),
                "end_frame": item.get("end_frame"),
                "start_seconds": item.get("start_seconds"),
                "end_seconds": item.get("end_seconds"),
                "source_status": item["source_status"],
            }
        )
        event_object_rows.extend(
            {
                "event_id": item["event_id"],
                "object_id": object_id_map[object_id],
            }
            for object_id in actor_ids
        )
    insert_rows("events", event_rows)
    insert_rows("event_objects", event_object_rows)

    geometry_rows = []
    for item in exports["geometries"]:
        source_object_id = item["object_id"]
        if source_object_id not in object_id_map:
            raise ValueError(
                f"Geometry {item['geometry_id']} references unknown object {source_object_id}"
            )
        geometry_rows.append(
            {
                "geometry_id": stable_id("geometry", video_id, item["geometry_id"]),
                "object_id": object_id_map[source_object_id],
                "source_geometry_id": item["geometry_id"],
                "frame_index": item["frame_index"],
                "timestamp_seconds": item["timestamp_seconds"],
                "label": item.get("label"),
                "confidence": item.get("confidence"),
                "spatial_position": item.get("spatial_position"),
                "bounding_box_pixels": item["bounding_box_pixels"],
            }
        )
    insert_rows("geometries", geometry_rows)

    return video_id, video_url, len(event_rows), len(event_object_rows), len(object_rows), len(geometry_rows)


run_started = perf_counter()
metadata_paths = sorted(PROCESSED_DIR.glob("*/meva_export/*.metadata.json"))
if not metadata_paths:
    raise FileNotFoundError(f"No metadata exports found under {PROCESSED_DIR}")

print(f"Found {len(metadata_paths)} video exports. Loading and validating...", flush=True)
video_exports = [load_exports(metadata_path) for metadata_path in metadata_paths]
for index, (video_name, exports) in enumerate(video_exports, start=1):
    print(f"Validating {index}/{len(video_exports)}: {video_name}", flush=True)
    validate_exports(video_name, exports)

create_tables()
reset_tables()
for index, (video_name, exports) in enumerate(video_exports, start=1):
    print(f"\nProcessing {index}/{len(video_exports)}: {video_name}", flush=True)
    video_id, video_url, event_count, event_object_count, object_count, geometry_count = sync_video(video_name, exports)
    print(
        f"Synced {video_name} ({video_id[:12]}…): {event_count} events, "
        f"{event_object_count} event-object links, {object_count} objects, {geometry_count} geometries\n{video_url}"
    )

print(f"\nCompleted in {perf_counter() - run_started:.1f} seconds.", flush=True)


Found 10 video exports. Loading and validating...
Validating 1/10: 2018-03-05.13-15-00.13-20-00.bus.G340
Validating 2/10: 2018-03-05.13-15-00.13-20-00.bus.G505
Validating 3/10: 2018-03-05.13-15-00.13-20-00.bus.G506
Validating 4/10: 2018-03-05.13-15-00.13-20-00.hospital.G341
Validating 5/10: 2018-03-05.13-15-00.13-20-00.school.G423
Validating 6/10: 2018-03-05.13-15-00.13-20-00.school.G424
Validating 7/10: 2018-03-05.13-15-01.13-20-01.school.G328
Validating 8/10: 2018-03-05.13-20-00.13-25-00.bus.G340
Validating 9/10: 2018-03-05.13-20-00.13-25-00.school.G336
Validating 10/10: 2018-03-05.13-20-00.13-25-00.school.G423
Creating the bronze schema and tables; removing legacy public tables...
Clearing existing bronze table rows...

Processing 1/10: 2018-03-05.13-15-00.13-20-00.bus.G340
  R2 already has original-videos/4b638634d856bbec2f837538aadeef51cdf7ee0455381af39aab6703f40443a5.mp4; skipping convert and upload
  Inserting 1 rows into bronze.videos...
    videos: 1/1 rows
  Inserting 20 rows